<a href="https://colab.research.google.com/github/cpython-projects/da_27_07_2026/blob/main/lesson_18.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Групування та зведені таблиці: `groupby`, `agg`, `pivot_table`, `crosstab`, `resample`, `cut`

**Навіщо це потрібно.** Досі ми рахували статистики по всьому датасету одразу ("яка середня ціна?"). Але 90% реальних бізнес-питань — це питання **по розрізах**: яка середня ціна по районах, скільки оголошень по місяцях, як ціна залежить від стану будинку й кількості спалень одночасно. Ця лекція — про інструменти pandas, якими аналітик рахує метрики **в розрізі** категорій, часу й числових діапазонів, а не по всьому датасету одразу.

In [42]:
import pandas as pd
import plotly.express as px
import seaborn as sns
import matplotlib.pyplot as plt

import warnings
warnings.filterwarnings('ignore')

In [43]:
df = pd.read_csv('https://raw.githubusercontent.com/cpython-projects/python_da_17_03_26/refs/heads/main/kc_house_data.csv')
df.head()

,id,date,price,bedrooms,bathrooms,sqft_living,sqft_lot,floors,waterfront,view,...,grade,sqft_above,sqft_basement,yr_built,yr_renovated,zipcode,lat,long,sqft_living15,sqft_lot15
0,7129300520,20141013T000000,221900.0,3,1.00,1180,5650,1.0,0,0,...,7,1180.0,0,1955,0,98178,47.5112,-122.257,1340,5650
1,6414100192,20141209T000000,538000.0,3,2.25,2570,7242,2.0,0,0,...,7,2170.0,400,1951,1991,98125,47.7210,-122.319,1690,7639
2,5631500400,20150225T000000,180000.0,2,1.00,770,10000,1.0,0,0,...,6,770.0,0,1933,0,98028,47.7379,-122.233,2720,8062
3,2487200875,20141209T000000,604000.0,4,3.00,1960,5000,1.0,0,0,...,7,1050.0,910,1965,0,98136,47.5208,-122.393,1360,5000
4,1954400510,20150218T000000,510000.0,3,2.00,1680,8080,1.0,0,0,...,8,1680.0,0,1987,0,98074,47.6168,-122.045,1800,7503


In [44]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 21613 entries, 0 to 21612
Data columns (total 21 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   id             21613 non-null  int64  
 1   date           21613 non-null  object 
 2   price          21613 non-null  float64
 3   bedrooms       21613 non-null  int64  
 4   bathrooms      21613 non-null  float64
 5   sqft_living    21613 non-null  int64  
 6   sqft_lot       21613 non-null  int64  
 7   floors         21613 non-null  float64
 8   waterfront     21613 non-null  int64  
 9   view           21613 non-null  int64  
 10  condition      21613 non-null  int64  
 11  grade          21613 non-null  int64  
 12  sqft_above     21611 non-null  float64
 13  sqft_basement  21613 non-null  int64  
 14  yr_built       21613 non-null  int64  
 15  yr_renovated   21613 non-null  int64  
 16  zipcode        21613 non-null  int64  
 17  lat            21613 non-null  float64
 18  long  

In [45]:
df['zipcode'] = df['zipcode'].astype('category')

# Групування даних з `groupby()`

## Що таке `groupby()`

Метод `groupby()` дозволяє згрупувати дані за одним або кількома критеріями для подальшого узагальнення (агрегації).

**Загальна формула:**

```python
df.groupby(ключ)[поля].агрегуюча_функція()
```

## Сценарії використання

In [46]:
df.columns.to_list()

['id',
 'date',
 'price',
 'bedrooms',
 'bathrooms',
 'sqft_living',
 'sqft_lot',
 'floors',
 'waterfront',
 'view',
 'condition',
 'grade',
 'sqft_above',
 'sqft_basement',
 'yr_built',
 'yr_renovated',
 'zipcode',
 'lat',
 'long',
 'sqft_living15',
 'sqft_lot15']

In [47]:
df.condition.unique()

array([3, 5, 4, 1, 2])

In [48]:
df['condition'] = pd.Categorical(df['condition'], categories=[1, 2, 3, 4, 5], ordered=True)

In [49]:
# Групування по одному полю — одна метрика
res = df.groupby('condition', observed=True)['price'].count()
res

,price
condition,
1,30
2,172
3,14031
4,5679
5,1701


In [50]:
type(res)

pandas.core.series.Series

In [51]:
# Групування по одному полю — кілька метрик
df.groupby('condition', observed=True)['price'].agg(['mean', 'median'])

,mean,median
condition,,
1,334431.666667,262500.0
2,327287.145349,279000.0
3,542012.578148,450000.0
4,521200.390033,440000.0
5,612418.089359,526000.0


## ⚠️ Пастка: `groupby()` по категоріальній колонці й параметр `observed`

Ми щойно перетворили `condition` на **категоріальний** (`Categorical`) тип. Це змінює поведінку `groupby()` непомітним чином. Подивимось, що станеться, якщо згрупувати одразу по двох полях — категоріальному `condition` і звичайному `bedrooms`:

In [52]:
df.bedrooms.unique()

array([ 3,  2,  4,  5,  1,  6,  7,  0,  8,  9, 11, 10, 33])

In [53]:
df.groupby(['condition', 'bedrooms'], observed=True)['price'].mean()

condition  bedrooms
1          0           1.420000e+05
           1           2.855000e+05
           2           2.772917e+05
           3           4.007438e+05
           4           4.457500e+05
           6           4.325000e+05
2          0           2.280000e+05
           1           2.450000e+05
           2           2.759067e+05
           3           2.963365e+05
           4           4.733883e+05
           5           2.355000e+05
           6           5.248167e+05
3          0           4.813600e+05
           1           3.135745e+05
           2           4.012794e+05
           3           4.671847e+05
           4           6.335218e+05
           5           8.130543e+05
           6           7.866955e+05
           7           9.179944e+05
           8           7.920000e+05
           9           8.939998e+05
           10          1.148000e+06
           11          5.200000e+05
4          0           1.399500e+05
           1           3.480948e+05
           2           4.059846e+05
           3           4.543136e+05
           4           6.159941e+05
           5           7.080780e+05
           6           8.747552e+05
           7           8.505730e+05
           8           9.200000e+05
           10          6.550000e+05
5          1           3.151792e+05
           2           4.250840e+05
           3           5.191837e+05
           4           7.181036e+05
           5           8.269887e+05
           6           9.623078e+05
           7           1.385000e+06
           8           2.635000e+06
           33          6.400000e+05
Name: price, dtype: float64

In [54]:
# Групування по кількох полях (без observed) — навмисно залишаємо типову помилку
df.groupby(['condition', 'bedrooms'])['price'].mean()

condition  bedrooms
1          0           1.420000e+05
           1           2.855000e+05
           2           2.772917e+05
           3           4.007438e+05
           4           4.457500e+05
                           ...     
5          8           2.635000e+06
           9                    NaN
           10                   NaN
           11                   NaN
           33          6.400000e+05
Name: price, Length: 65, dtype: float64

Зверніть увагу на результат: з'являються рядки типу `(condition=1, bedrooms=5) → NaN`. Причина — параметр `observed`:

> 💡 На початку ноутбука ми викликали `warnings.filterwarnings('ignore')`, тож `FutureWarning`, який pandas насправді видає для цього коду, тут не показується. Якщо прибрати цей рядок (або виконати код в окремій клітинці без придушення попереджень), побачите: `FutureWarning: The default of observed=False is deprecated...`.

* `observed=False` (поточна поведінка за замовчуванням, але **вже застаріла** і найближчим часом зміниться) — `groupby()` будує **повний перетин** усіх категорій `Categorical`-колонки з усіма значеннями другої колонки, навіть якщо такої комбінації жодного разу не було в даних. Звідси — "зайві" рядки з `NaN`.
* `observed=True` — повертає **тільки ті комбінації, які реально зустрічаються** в даних, як і для звичайних (не категоріальних) колонок.

Порівняємо розмір результату:

In [55]:
# .size() — просто кількість рядків у кожній групі (не плутати з .count(),
# яка рахує непорожні значення по кожній колонці окремо)
n_observed_false = len(df.groupby(['condition', 'bedrooms']).size())
n_observed_true = len(df.groupby(['condition', 'bedrooms'], observed=True).size())

print('Комбінацій з observed=False (включно з неіснуючими):', n_observed_false)
print('Комбінацій з observed=True (лише реальні):', n_observed_true)

Комбінацій з observed=False (включно з неіснуючими): 65
Комбінацій з observed=True (лише реальні): 44


**Правило:** якщо серед колонок групування є хоч одна `Categorical` — завжди явно пишіть `observed=True`, якщо не маєте конкретної причини бачити й порожні комбінації категорій. Інакше результат може "роздутися" зайвими рядками з `NaN`, які легко сплутати з реальними пропусками в даних. Далі в цій лекції скрізь, де групуємо по `condition`, `grade` чи `zipcode` (усі вони `Categorical`), додаємо `observed=True`.

In [56]:
# Групування по кількох полях — коректна версія
df.groupby(['condition', 'bedrooms'], observed=True)['price'].mean()

condition  bedrooms
1          0           1.420000e+05
           1           2.855000e+05
           2           2.772917e+05
           3           4.007438e+05
           4           4.457500e+05
           6           4.325000e+05
2          0           2.280000e+05
           1           2.450000e+05
           2           2.759067e+05
           3           2.963365e+05
           4           4.733883e+05
           5           2.355000e+05
           6           5.248167e+05
3          0           4.813600e+05
           1           3.135745e+05
           2           4.012794e+05
           3           4.671847e+05
           4           6.335218e+05
           5           8.130543e+05
           6           7.866955e+05
           7           9.179944e+05
           8           7.920000e+05
           9           8.939998e+05
           10          1.148000e+06
           11          5.200000e+05
4          0           1.399500e+05
           1           3.480948e+05
           2           4.059846e+05
           3           4.543136e+05
           4           6.159941e+05
           5           7.080780e+05
           6           8.747552e+05
           7           8.505730e+05
           8           9.200000e+05
           10          6.550000e+05
5          1           3.151792e+05
           2           4.250840e+05
           3           5.191837e+05
           4           7.181036e+05
           5           8.269887e+05
           6           9.623078e+05
           7           1.385000e+06
           8           2.635000e+06
           33          6.400000e+05
Name: price, dtype: float64

In [57]:
# Агрегування кількох полів — кілька метрик
df.groupby(['bedrooms', 'condition'], observed=True)['price'].agg(['mean', 'median'])

mean     median
bedrooms condition                         
0        1          1.420000e+05   142000.0
         2          2.280000e+05   228000.0
         3          4.813600e+05   329975.0
         4          1.399500e+05   139950.0
1        1          2.855000e+05   221000.0
         2          2.450000e+05   250000.0
         3          3.135745e+05   309975.0
         4          3.480948e+05   308475.0
         5          3.151792e+05   275500.0
2        1          2.772917e+05   245000.0
         2          2.759067e+05   255000.0
         3          4.012794e+05   371000.0
         4          4.059846e+05   389050.0
         5          4.250840e+05   391650.0
3        1          4.007438e+05   261000.0
         2          2.963365e+05   235000.0
         3          4.671847e+05   415700.0
         4          4.543136e+05   395000.0
         5          5.191837e+05   456075.0
4        1          4.457500e+05   434000.0
         2          4.733883e+05   375000.0
         3          6.335218e+05   549950.0
         4          6.159941e+05   530000.0
         5          7.181036e+05   620000.0
5        2          2.355000e+05   235500.0
         3          8.130543e+05   633000.0
         4          7.080780e+05   573725.0
         5          8.269887e+05   660500.0
6        1          4.325000e+05   432500.0
         2          5.248167e+05   619500.0
         3          7.866955e+05   652500.0
         4          8.747552e+05   611000.0
         5          9.623078e+05   735000.0
7        3          9.179944e+05   727160.0
         4          8.505730e+05   597157.0
         5          1.385000e+06   805000.0
8        3          7.920000e+05   637500.0
         4          9.200000e+05   680000.0
         5          2.635000e+06  2635000.0
9        3          8.939998e+05   817000.0
10       3          1.148000e+06  1148000.0
         4          6.550000e+05   655000.0
11       3          5.200000e+05   520000.0
33       5          6.400000e+05   640000.0

## `reset_index()`

Після `groupby()` індексом стає групувальне поле (або поля). Щоб зробити з них **звичайні колонки**, потрібно:

```python
.reset_index()
```

Інакше графіки чи подальша робота будуть ускладнені.

> 💡 Альтернатива: одразу передати `as_index=False` у сам `groupby()` — `df.groupby('condition', observed=True, as_index=False)['price'].mean()` дає той самий результат за одну дію, без окремого `.reset_index()`.

In [58]:
res = df.groupby('condition', observed=True)['price'].agg(['mean', 'median']).reset_index()

In [59]:
res

,condition,mean,median
0,1,334431.666667,262500.0
1,2,327287.145349,279000.0
2,3,542012.578148,450000.0
3,4,521200.390033,440000.0
4,5,612418.089359,526000.0


In [60]:
res = df.groupby('condition', observed=True, as_index=False)['price'].agg(['mean', 'median'])
res

,condition,mean,median
0,1,334431.666667,262500.0
1,2,327287.145349,279000.0
2,3,542012.578148,450000.0
3,4,521200.390033,440000.0
4,5,612418.089359,526000.0


## Задача: обчислити:

* середню ціну (`price`)
* кількість оголошень (`count`)
* медіанну площу (`sqft_living`)
  по кожному району

In [61]:
df.groupby('zipcode', observed=True).agg({
    'price': ['mean', 'count'],
    'sqft_living': 'median'
}).reset_index()

zipcode         price       sqft_living
                    mean count      median
0    98001  2.808047e+05   362      1822.0
1    98002  2.342840e+05   199      1570.0
2    98003  2.941113e+05   280      1780.0
3    98004  1.355927e+06   317      2660.0
4    98005  8.101649e+05   168      2505.0
..     ...           ...   ...         ...
65   98177  6.761854e+05   255      2120.0
66   98178  3.106128e+05   262      1725.0
67   98188  2.890783e+05   136      1690.0
68   98198  3.028789e+05   280      1610.0
69   98199  7.918208e+05   317      1980.0

[70 rows x 4 columns]

In [62]:
df.groupby('zipcode', observed=True)[['price', 'sqft_living']].agg(['mean', 'count', 'median']).reset_index()

zipcode         price                   sqft_living              
                    mean count     median         mean count  median
0    98001  2.808047e+05   362   260000.0  1900.856354   362  1822.0
1    98002  2.342840e+05   199   235000.0  1627.743719   199  1570.0
2    98003  2.941113e+05   280   267475.0  1928.882143   280  1780.0
3    98004  1.355927e+06   317  1150000.0  2909.022082   317  2660.0
4    98005  8.101649e+05   168   765475.0  2656.803571   168  2505.0
..     ...           ...   ...        ...          ...   ...     ...
65   98177  6.761854e+05   255   554000.0  2323.333333   255  2120.0
66   98178  3.106128e+05   262   278277.0  1729.351145   262  1725.0
67   98188  2.890783e+05   136   264000.0  1802.772059   136  1690.0
68   98198  3.028789e+05   280   265000.0  1745.360714   280  1610.0
69   98199  7.918208e+05   317   689800.0  2161.798107   317  1980.0

[70 rows x 7 columns]

# Агрегації `.agg()` в Pandas

## Навіщо потрібні агрегації?

Агрегація — це зведення (узагальнення) числових значень:

| Приклади запитань | Агрегація |
|---|---|
| Яка середня площа будинків в районі Х? | `mean()` |
| Скільки будинків виставлено у районі Y? | `count()` |
| Яка максимальна ціна у 1-bedroom будинків? | `max()` |
| Яка медіанна площа кухні у 3-bedroom будинків? | `median()` |

---

## Синтаксис `.agg()` — на одному або кількох полях

```python
df.agg({'col1': 'mean', 'col2': 'max'})
```

Або на одній колонці:

```python
df['price'].agg(['mean', 'median', 'std'])
```

---

## Комбінування з `groupby()`

```python
df.groupby('zipcode', observed=True).agg({
    'price': ['mean', 'median', 'min', 'max'],
    'sqft_living': 'mean'
})
```

У результаті — **ієрархічний заголовок колонок (MultiIndex)** → бажано `.reset_index()` та перейменувати колонки.

In [63]:
summary = df[['price', 'sqft_living']].agg(
    ['mean', 'median', 'std', 'min', 'max', 'count']
).round(1).reset_index()

summary

,index,price,sqft_living
0,mean,540088.1,2079.9
1,median,450000.0,1910.0
2,std,367127.2,918.4
3,min,75000.0,290.0
4,max,7700000.0,13540.0
5,count,21613.0,21613.0


In [64]:
# .T — транспонування: міняє місцями рядки й колонки.
# Тут робимо це, щоб "price" і "sqft_living" стали рядками,
# а назви статистик (mean, median, ...) — колонками: так таблицю зручніше читати,
# коли показників (6) більше, ніж колонок (2).

summary = df[['price', 'sqft_living']].agg(
    ['mean', 'median', 'std', 'min', 'max', 'count']
).T.round(1)

summary

,mean,median,std,min,max,count
price,540088.1,450000.0,367127.2,75000.0,7700000.0,21613.0
sqft_living,2079.9,1910.0,918.4,290.0,13540.0,21613.0


## Нюанси

| Питання | Коментар |
|---|---|
| `NaN` | Ігноруються у функціях агрегування (`skipna=True` за замовчуванням) |
| `.agg()` після `groupby()` | Повертає MultiIndex → `.reset_index()` |
| `std`, `min`, `max`, `median` | Обчислюються лише для числових колонок |
| Для bool | Можна використовувати `mean()` — частка `True` |
| `groupby()` по `Categorical`-колонці | Завжди додавайте `observed=True` (див. застереження вище) |

---

# Зведені таблиці (`pivot_table`)

## Що таке `pivot_table`

`pivot_table()` — це розширений варіант `groupby`, який дозволяє:

* Зробити зведену таблицю по кількох категоріях
* Розмістити одну категорію по рядках, іншу по стовпцях
* Одразу застосувати агрегуючу функцію
* Аналог Excel PivotTable!

---

## Синтаксис

```python
pd.pivot_table(
    data=df,
    values='метрика',
    index='рядки',
    columns='стовпці',
    aggfunc='функція'
)
```

In [65]:
# Приклад: середня ціна за кількістю спалень і станом будинку (condition)
df.groupby(['bedrooms', 'condition'], observed=True)['price'].mean()

bedrooms  condition
0         1            1.420000e+05
          2            2.280000e+05
          3            4.813600e+05
          4            1.399500e+05
1         1            2.855000e+05
          2            2.450000e+05
          3            3.135745e+05
          4            3.480948e+05
          5            3.151792e+05
2         1            2.772917e+05
          2            2.759067e+05
          3            4.012794e+05
          4            4.059846e+05
          5            4.250840e+05
3         1            4.007438e+05
          2            2.963365e+05
          3            4.671847e+05
          4            4.543136e+05
          5            5.191837e+05
4         1            4.457500e+05
          2            4.733883e+05
          3            6.335218e+05
          4            6.159941e+05
          5            7.181036e+05
5         2            2.355000e+05
          3            8.130543e+05
          4            7.080780e+05
          5            8.269887e+05
6         1            4.325000e+05
          2            5.248167e+05
          3            7.866955e+05
          4            8.747552e+05
          5            9.623078e+05
7         3            9.179944e+05
          4            8.505730e+05
          5            1.385000e+06
8         3            7.920000e+05
          4            9.200000e+05
          5            2.635000e+06
9         3            8.939998e+05
10        3            1.148000e+06
          4            6.550000e+05
11        3            5.200000e+05
33        5            6.400000e+05
Name: price, dtype: float64

Той самий результат зручніше читати як зведену таблицю — `bedrooms` по рядках, `condition` по стовпцях:

In [66]:
res = pd.pivot_table(
    data=df[df.bedrooms <= 11],
    values='price',
    index='bedrooms',
    columns='condition',
    aggfunc=['median', 'mean'],
)

In [67]:
res

median                                                     mean  \
condition         1         2          3         4          5              1   
bedrooms                                                                       
0          142000.0  228000.0   329975.0  139950.0        NaN  142000.000000   
1          221000.0  250000.0   309975.0  308475.0   275500.0  285500.000000   
2          245000.0  255000.0   371000.0  389050.0   391650.0  277291.666667   
3          261000.0  235000.0   415700.0  395000.0   456075.0  400743.750000   
4          434000.0  375000.0   549950.0  530000.0   620000.0  445750.000000   
5               NaN  235500.0   633000.0  573725.0   660500.0            NaN   
6          432500.0  619500.0   652500.0  611000.0   735000.0  432500.000000   
7               NaN       NaN   727160.0  597157.0   805000.0            NaN   
8               NaN       NaN   637500.0  680000.0  2635000.0            NaN   
9               NaN       NaN   817000.0       NaN        NaN            NaN   
10              NaN       NaN  1148000.0  655000.0        NaN            NaN   
11              NaN       NaN   520000.0       NaN        NaN            NaN   

                                                                     
condition              2             3              4             5  
bedrooms                                                             
0          228000.000000  4.813600e+05  139950.000000           NaN  
1          245000.000000  3.135745e+05  348094.791667  3.151792e+05  
2          275906.725490  4.012794e+05  405984.607242  4.250840e+05  
3          296336.463768  4.671847e+05  454313.579860  5.191837e+05  
4          473388.333333  6.335218e+05  615994.060048  7.181036e+05  
5          235500.000000  8.130543e+05  708077.956938  8.269887e+05  
6          524816.666667  7.866955e+05  874755.172414  9.623078e+05  
7                    NaN  9.179944e+05  850573.000000  1.385000e+06  
8                    NaN  7.920000e+05  920000.000000  2.635000e+06  
9                    NaN  8.939998e+05            NaN           NaN  
10                   NaN  1.148000e+06  655000.000000           NaN  
11                   NaN  5.200000e+05            NaN           NaN

## Додаткові параметри

| Параметр | Значення |
|---|---|
| `fill_value` | чим заповнити `NaN` (комбінації без жодного спостереження) |
| `margins=True` | додати підсумковий рядок/колонку "Усього" |
| `margins_name` | назва підсумкового рядка/колонки (за замовчуванням `'All'`) |
| `aggfunc=['mean', 'median']` | кілька функцій одразу |

Застосуємо одразу всі чотири — порожні комбінації заповнимо нулем, і додамо підсумки:

In [68]:
pd.pivot_table(
    data=df[df.bedrooms <= 11],
    values='price',
    index='bedrooms',
    columns='condition',
    aggfunc='median',
    fill_value=0,
    margins=True,
    margins_name='Усього',
)

condition,1,2,3,4,5,Усього
bedrooms,,,,,,
0,142000.0,228000.0,329975.0,139950.0,0.0,288000.0
1,221000.0,250000.0,309975.0,308475.0,275500.0,299000.0
2,245000.0,255000.0,371000.0,389050.0,391650.0,374000.0
3,261000.0,235000.0,415700.0,395000.0,456075.0,413000.0
4,434000.0,375000.0,549950.0,530000.0,620000.0,549997.5
5,0.0,235500.0,633000.0,573725.0,660500.0,620000.0
6,432500.0,619500.0,652500.0,611000.0,735000.0,650000.0
7,0.0,0.0,727160.0,597157.0,805000.0,728580.0
8,0.0,0.0,637500.0,680000.0,2635000.0,700000.0


> 💡 `fill_value=0` тут зручний лише тому, що жодна реальна ціна не дорівнює нулю, тож нуль однозначно читається як "даних немає", а не як справжнє значення. Для метрики, де 0 — цілком реальне число (наприклад, кількість переоцінок), так підміняти `NaN` не можна: 0 і "немає даних" — це різні речі.

# Частотні таблиці — `pd.crosstab()`

## Що таке `crosstab`

`pd.crosstab()` — це спосіб підрахунку **кількості випадків** для комбінацій категоріальних змінних.

## Синтаксис

```python
pd.crosstab(index=df['рядки'], columns=df['стовпці'])
```

In [69]:
df.columns.to_list()

['id',
 'date',
 'price',
 'bedrooms',
 'bathrooms',
 'sqft_living',
 'sqft_lot',
 'floors',
 'waterfront',
 'view',
 'condition',
 'grade',
 'sqft_above',
 'sqft_basement',
 'yr_built',
 'yr_renovated',
 'zipcode',
 'lat',
 'long',
 'sqft_living15',
 'sqft_lot15']

In [70]:
df['is_renovated'] = df['yr_renovated'] > 0
df

,id,date,price,bedrooms,bathrooms,sqft_living,sqft_lot,floors,waterfront,view,...,sqft_above,sqft_basement,yr_built,yr_renovated,zipcode,lat,long,sqft_living15,sqft_lot15,is_renovated
0,7129300520,20141013T000000,221900.0,3,1.00,1180,5650,1.0,0,0,...,1180.0,0,1955,0,98178,47.5112,-122.257,1340,5650,False
1,6414100192,20141209T000000,538000.0,3,2.25,2570,7242,2.0,0,0,...,2170.0,400,1951,1991,98125,47.7210,-122.319,1690,7639,True
2,5631500400,20150225T000000,180000.0,2,1.00,770,10000,1.0,0,0,...,770.0,0,1933,0,98028,47.7379,-122.233,2720,8062,False
3,2487200875,20141209T000000,604000.0,4,3.00,1960,5000,1.0,0,0,...,1050.0,910,1965,0,98136,47.5208,-122.393,1360,5000,False
4,1954400510,20150218T000000,510000.0,3,2.00,1680,8080,1.0,0,0,...,1680.0,0,1987,0,98074,47.6168,-122.045,1800,7503,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
21608,263000018,20140521T000000,360000.0,3,2.50,1530,1131,3.0,0,0,...,1530.0,0,2009,0,98103,47.6993,-122.346,1530,1509,False
21609,6600060120,20150223T000000,400000.0,4,2.50,2310,5813,2.0,0,0,...,2310.0,0,2014,0,98146,47.5107,-122.362,1830,7200,False
21610,1523300141,20140623T000000,402101.0,2,0.75,1020,1350,2.0,0,0,...,1020.0,0,2009,0,98144,47.5944,-122.299,1020,2007,False
21611,291310100,20150116T000000,400000.0,3,2.50,1600,2388,2.0,0,0,...,1600.0,0,2004,0,98027,47.5345,-122.069,1410,1287,False


In [71]:
df.grade.unique()

array([ 7,  6,  8, 11,  9,  5, 10, 12,  4,  3, 13,  1])

In [72]:
df['grade'] = pd.Categorical(df['grade'], categories=[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13], ordered=True)

In [73]:
# Приклад: скільки якісних будинків було оновлено
pd.crosstab(index=df['grade'], columns=df['is_renovated'])

is_renovated,False,True
grade,,
1,1,0
3,3,0
4,28,1
5,236,6
6,1941,97
7,8645,336
8,5803,265
9,2477,138
10,1084,50


> 💡 Зверніть увагу: тут **не** потрібен `observed=True` — на відміну від `groupby()`, `crosstab()` за замовчуванням показує лише ті категорії, які реально зустрічаються в даних (наприклад, `grade=2` в таблиці немає, бо жодного будинку з такою оцінкою в датасеті не існує). Пастка з `observed` стосується саме `groupby()`.

## Нюанси

| Питання | Коментар |
|---|---|
| `NaN` у категоріях | не враховуються |
| `normalize='index'` | показує **частки** замість кількостей (по рядках) |
| `normalize='columns'` | те саме, але частки по колонках |
| Можна додати `values=`, `aggfunc=` | як у `pivot_table` |

---

In [74]:
# Ціновий сегмент × кількість спалень
df['price_segment'] = pd.qcut(
    df['price'],
    q=4,
    labels=['Low', 'Mid-Low', 'Mid-High', 'High']
)

pd.crosstab(
    df['price_segment'],
    df['bedrooms'],
)

bedrooms,0,1,2,3,4,5,6,7,8,9,10,11,33
price_segment,,,,,,,,,,,,,
Low,8,116,1030,3027,1033,158,29,3,0,0,0,0,0
Mid-Low,3,56,859,2714,1501,273,45,5,3,1,0,0,0
Mid-High,0,21,631,2470,1767,413,59,10,2,1,0,1,1
High,2,6,240,1613,2581,757,139,20,8,4,3,0,0


In [75]:
pd.crosstab(
    df['price_segment'],
    df['bedrooms'],
    normalize='index'
) * 100

bedrooms,0,1,2,3,4,5,6,7,8,9,10,11,33
price_segment,,,,,,,,,,,,,
Low,0.148038,2.146558,19.059956,56.014064,19.115470,2.923760,0.536640,0.055514,0.000000,0.000000,0.000000,0.000000,0.000000
Mid-Low,0.054945,1.025641,15.732601,49.706960,27.490842,5.000000,0.824176,0.091575,0.054945,0.018315,0.000000,0.000000,0.000000
Mid-High,0.000000,0.390625,11.737351,45.944940,32.868304,7.682292,1.097470,0.186012,0.037202,0.018601,0.000000,0.018601,0.018601
High,0.037223,0.111669,4.466778,30.020473,48.036479,14.088963,2.587009,0.372232,0.148893,0.074446,0.055835,0.000000,0.000000


Останню таблицю зручно читати так: у кожній колонці (кількість спалень) — на які відсотки розкладаються цінові сегменти. Наприклад, видно, яка частка 2-кімнатних будинків потрапляє в дешевий (`Low`) сегмент, а яка — у дорогий (`High`).

# `resample()` — агрегування за датами

## Що таке `resample`

`resample()` — це аналог `groupby()` **по часу**, який працює з колонкою `datetime`.

Необхідно:

* мати колонку типу `datetime64[ns]`
* встановити її як **індекс**

---

## Синтаксис:

```python
df.resample('період')['метрика'].agg(функція)
```

### Базові часові одиниці

| Частота | Alias | Опис |
|---|---|---|
| Day | `'D'` | календарний день |
| Hour | `'h'` | година |
| Minute | `'min'` | хвилина |
| Second | `'s'` | секунда |
| Millisecond | `'ms'` | мілісекунда |
| Microsecond | `'us'` | мікросекунда |
| Nanosecond | `'ns'` | наносекунда |

> ⚠️ У старіших підручниках і на Stack Overflow часто трапляються написання великими літерами: `'H'`, `'T'`/`'min'`, `'S'`. Вони досі технічно працюють, але вже видають `FutureWarning` і будуть прибрані в майбутніх версіях pandas. У коді нижче й надалі використовуємо **актуальні** (малі літери) варіанти.

---

### Бізнес-час і бізнес-дні

| Частота | Alias | Опис |
|---|---|---|
| BDay / BusinessDay | `'B'` | бізнес-день (пн–пт) |
| CDay / CustomBusinessDay | `'C'` | кастомний бізнес-день (за користувацьким календарем) |
| BusinessHour | `'bh'` | бізнес-година |
| CustomBusinessHour | `'cbh'` | кастомна бізнес-година |

---

### Тижні

| Частота | Alias | Опис |
|---|---|---|
| Week | `'W'` | тиждень, можна якір на день (`'W-MON'`, `'W-FRI'`) |

**Приклади**

* `'W'` — тиждень, кінець у неділю
* `'W-MON'` — тиждень з кінцем у понеділок

---

### Місячні частоти

| Частота | Alias | Опис |
|---|---|---|
| MonthEnd | `'ME'` | кінець календарного місяця |
| MonthBegin | `'MS'` | початок місяця |
| BMonthEnd | `'BME'` | бізнес-кінець місяця |
| BMonthBegin | `'BMS'` | бізнес-початок місяця |

> ⚠️ Стара назва `'M'` (без `E`) означала те саме, що й `'ME'`, але вже видає `FutureWarning` — у коді використовуємо `'ME'`.

---

### Квартали

| Частота | Alias | Опис |
|---|---|---|
| QuarterEnd | `'QE'` | кінець календарного кварталу |
| QuarterBegin | `'QS'` | початок кварталу |

> ⚠️ Так само: стара назва `'Q'` → використовуємо `'QE'`.

In [76]:
# Підготовка даних
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 21613 entries, 0 to 21612
Data columns (total 23 columns):
 #   Column         Non-Null Count  Dtype   
---  ------         --------------  -----   
 0   id             21613 non-null  int64   
 1   date           21613 non-null  object  
 2   price          21613 non-null  float64 
 3   bedrooms       21613 non-null  int64   
 4   bathrooms      21613 non-null  float64 
 5   sqft_living    21613 non-null  int64   
 6   sqft_lot       21613 non-null  int64   
 7   floors         21613 non-null  float64 
 8   waterfront     21613 non-null  int64   
 9   view           21613 non-null  int64   
 10  condition      21613 non-null  category
 11  grade          21613 non-null  category
 12  sqft_above     21611 non-null  float64 
 13  sqft_basement  21613 non-null  int64   
 14  yr_built       21613 non-null  int64   
 15  yr_renovated   21613 non-null  int64   
 16  zipcode        21613 non-null  category
 17  lat            21613 non-null  

In [77]:
df.date.unique()

array(['20141013T000000', '20141209T000000', '20150225T000000',
       '20150218T000000', '20140512T000000', '20140627T000000',
       '20150115T000000', '20150415T000000', '20150312T000000',
       '20150403T000000', '20140527T000000', '20140528T000000',
       '20141007T000000', '20150124T000000', '20140731T000000',
       '20140529T000000', '20141205T000000', '20150424T000000',
       '20140514T000000', '20140826T000000', '20140703T000000',
       '20140516T000000', '20141120T000000', '20141103T000000',
       '20140626T000000', '20141201T000000', '20140624T000000',
       '20150302T000000', '20141110T000000', '20141203T000000',
       '20140613T000000', '20141230T000000', '20150213T000000',
       '20140620T000000', '20140715T000000', '20140811T000000',
       '20140707T000000', '20141028T000000', '20140729T000000',
       '20140718T000000', '20150325T000000', '20140716T000000',
       '20150428T000000', '20150311T000000', '20140916T000000',
       '20150217T000000', '20141231T0000

In [78]:
df['date'] = pd.to_datetime(df['date'], format='%Y%m%dT%H%M%S', errors='coerce')

In [79]:
df

,id,date,price,bedrooms,bathrooms,sqft_living,sqft_lot,floors,waterfront,view,...,sqft_basement,yr_built,yr_renovated,zipcode,lat,long,sqft_living15,sqft_lot15,is_renovated,price_segment
0,7129300520,2014-10-13,221900.0,3,1.00,1180,5650,1.0,0,0,...,0,1955,0,98178,47.5112,-122.257,1340,5650,False,Low
1,6414100192,2014-12-09,538000.0,3,2.25,2570,7242,2.0,0,0,...,400,1951,1991,98125,47.7210,-122.319,1690,7639,True,Mid-High
2,5631500400,2015-02-25,180000.0,2,1.00,770,10000,1.0,0,0,...,0,1933,0,98028,47.7379,-122.233,2720,8062,False,Low
3,2487200875,2014-12-09,604000.0,4,3.00,1960,5000,1.0,0,0,...,910,1965,0,98136,47.5208,-122.393,1360,5000,False,Mid-High
4,1954400510,2015-02-18,510000.0,3,2.00,1680,8080,1.0,0,0,...,0,1987,0,98074,47.6168,-122.045,1800,7503,False,Mid-High
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
21608,263000018,2014-05-21,360000.0,3,2.50,1530,1131,3.0,0,0,...,0,2009,0,98103,47.6993,-122.346,1530,1509,False,Mid-Low
21609,6600060120,2015-02-23,400000.0,4,2.50,2310,5813,2.0,0,0,...,0,2014,0,98146,47.5107,-122.362,1830,7200,False,Mid-Low
21610,1523300141,2014-06-23,402101.0,2,0.75,1020,1350,2.0,0,0,...,0,2009,0,98144,47.5944,-122.299,1020,2007,False,Mid-Low
21611,291310100,2015-01-16,400000.0,3,2.50,1600,2388,2.0,0,0,...,0,2004,0,98027,47.5345,-122.069,1410,1287,False,Mid-Low


In [80]:
df.set_index('date', inplace=True)
df

,id,price,bedrooms,bathrooms,sqft_living,sqft_lot,floors,waterfront,view,condition,...,sqft_basement,yr_built,yr_renovated,zipcode,lat,long,sqft_living15,sqft_lot15,is_renovated,price_segment
date,,,,,,,,,,,,,,,,,,,,,
2014-10-13,7129300520,221900.0,3,1.00,1180,5650,1.0,0,0,3,...,0,1955,0,98178,47.5112,-122.257,1340,5650,False,Low
2014-12-09,6414100192,538000.0,3,2.25,2570,7242,2.0,0,0,3,...,400,1951,1991,98125,47.7210,-122.319,1690,7639,True,Mid-High
2015-02-25,5631500400,180000.0,2,1.00,770,10000,1.0,0,0,3,...,0,1933,0,98028,47.7379,-122.233,2720,8062,False,Low
2014-12-09,2487200875,604000.0,4,3.00,1960,5000,1.0,0,0,5,...,910,1965,0,98136,47.5208,-122.393,1360,5000,False,Mid-High
2015-02-18,1954400510,510000.0,3,2.00,1680,8080,1.0,0,0,3,...,0,1987,0,98074,47.6168,-122.045,1800,7503,False,Mid-High
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2014-05-21,263000018,360000.0,3,2.50,1530,1131,3.0,0,0,3,...,0,2009,0,98103,47.6993,-122.346,1530,1509,False,Mid-Low
2015-02-23,6600060120,400000.0,4,2.50,2310,5813,2.0,0,0,3,...,0,2014,0,98146,47.5107,-122.362,1830,7200,False,Mid-Low
2014-06-23,1523300141,402101.0,2,0.75,1020,1350,2.0,0,0,3,...,0,2009,0,98144,47.5944,-122.299,1020,2007,False,Mid-Low


In [81]:
df.sort_index(inplace=True)

In [82]:
df

,id,price,bedrooms,bathrooms,sqft_living,sqft_lot,floors,waterfront,view,condition,...,sqft_basement,yr_built,yr_renovated,zipcode,lat,long,sqft_living15,sqft_lot15,is_renovated,price_segment
date,,,,,,,,,,,,,,,,,,,,,
2014-05-02,5561000190,437500.0,3,2.25,1970,35100,2.0,0,0,4,...,0,1977,0,98027,47.4635,-121.991,2340,35100,False,Mid-Low
2014-05-02,472000620,790000.0,3,2.50,2600,4750,1.0,0,0,4,...,900,1951,0,98117,47.6833,-122.400,2380,4750,False,High
2014-05-02,1024069009,675000.0,5,2.50,2820,67518,2.0,0,0,3,...,0,1979,0,98029,47.5794,-122.025,2820,48351,False,High
2014-05-02,7853361370,555000.0,4,2.50,3310,6500,2.0,0,0,3,...,0,2012,0,98065,47.5150,-121.870,2380,5000,False,Mid-High
2014-05-02,5056500260,440000.0,4,2.25,2160,8119,1.0,0,0,3,...,1080,1966,0,98006,47.5443,-122.177,1850,9000,False,Mid-Low
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2015-05-14,1422700040,183000.0,3,1.00,1170,7320,1.0,0,0,3,...,0,1962,0,98188,47.4685,-122.282,2040,7320,False,Low
2015-05-14,8730000270,359000.0,2,2.75,1370,1140,2.0,0,0,3,...,290,2009,0,98133,47.7052,-122.343,1370,1090,False,Mid-Low
2015-05-15,7923600250,450000.0,5,2.00,1870,7344,1.5,0,0,3,...,0,1960,0,98007,47.5951,-122.144,1870,7650,False,Mid-Low


In [83]:
# Приклад: середня ціна будинка по місяцях
res = df.resample('ME')['price'].median()
res

,price
date,
2014-05-31,465000.0
2014-06-30,465000.0
2014-07-31,465000.0
2014-08-31,442100.0
2014-09-30,450000.0
2014-10-31,446900.0
2014-11-30,435000.0
2014-12-31,432500.0
2015-01-31,438500.0


In [99]:
df.resample('ME')['price'].mean()

,price
date,
2014-05-31,548080.276584
2014-06-30,558002.199541
2014-07-31,544788.764360
2014-08-31,536445.276804
2014-09-30,529253.821871
2014-10-31,539026.971778
2014-11-30,521961.009213
2014-12-31,524461.866757
2015-01-31,525870.889571


In [100]:
df.resample('QE')['price'].mean()

,price
date,
2014-06-30,553558.947315
2014-09-30,537405.582278
2014-12-31,529467.025840
2015-03-31,528655.415550
2015-06-30,561004.517205


In [86]:
res = df.resample('ME')['price'].mean()

In [87]:
# Візуалізація: динаміка середньої ціни по місяцях
fig = px.line(
    res,
    title='Середня ціна будинків по місяцях'
)
fig.show()

In [101]:
# Кількість угод по тижнях
res = df.resample('W')['price'].count()
res

,price
date,
2014-05-04,76
2014-05-11,429
2014-05-18,410
2014-05-25,484
2014-06-01,376
2014-06-08,456
2014-06-15,467
2014-06-22,537
2014-06-29,636


In [90]:
fig = px.line(
    res,
    title='Кількість угод по тижнях'
)
fig.show()

## Навіщо аналітику `resample`

| Що дізнаємось | Як це корисно |
|---|---|
| Сезонність ринку | порівняння цін по місяцях/роках |
| Активність користувачів | по кількості оголошень |
| Аналіз впливу подій (інфляція, війна) | тренди по кварталах |
| Маркетинг | коли зростає інтерес до нерухомості |

---

## Нюанси

| Проблема | Рішення |
|---|---|
| Не дата в індексі | `.set_index('date')` перед `resample()` |
| Пропущені періоди | Автоматично створюються `NaN` |
| Смугастий графік | Використовуйте `'ME'`, `'QE'`, а не `'D'` |
| Великі значення | Обрізати викиди перед побудовою |
| Старі аліаси (`'M'`, `'Q'`, `'H'`, `'T'`...) | Видають `FutureWarning` — замінюйте на `'ME'`, `'QE'`, `'h'`, `'min'` |

---

# `pd.cut()` — групування числових значень у категорії

## Що таке `cut`

`cut()` розбиває числову колонку на **інтервали (бінінг)**. Наприклад, можна створити категорії будинків:

* до 1000 м² → «малий»
* 1000–4000 м² → «середній»
* понад 4000 м² → «великий»

Це корисно для:

* **аналізу розподілу** (гістограми, pie charts)
* **порівнянь середніх цін між групами**
* **створення сегментів**

---

## Синтаксис

```python
pd.cut(Series, bins, labels=..., include_lowest=True)
```

| Параметр | Значення |
|---|---|
| `bins` | кількість або список меж |
| `labels` | імена категорій |
| `include_lowest` | включити нижню межу в перший бін |

---

In [113]:
# Приклад: розбиття будинків за площею
df['area_category'] = pd.cut(
    df['sqft_living'],
    bins=[900, 1000, 4000, df.sqft_living.max()],
    labels=['малий', 'середній', 'великий'],
    include_lowest=True
)

In [108]:
df

,id,price,bedrooms,bathrooms,sqft_living,sqft_lot,floors,waterfront,view,condition,...,yr_renovated,zipcode,lat,long,sqft_living15,sqft_lot15,is_renovated,price_segment,area_category,area_auto_category
date,,,,,,,,,,,,,,,,,,,,,
2014-05-02,5561000190,437500.0,3,2.25,1970,35100,2.0,0,0,4,...,0,98027,47.4635,-121.991,2340,35100,False,Mid-Low,середній,середня
2014-05-02,472000620,790000.0,3,2.50,2600,4750,1.0,0,0,4,...,0,98117,47.6833,-122.400,2380,4750,False,High,середній,велика
2014-05-02,1024069009,675000.0,5,2.50,2820,67518,2.0,0,0,3,...,0,98029,47.5794,-122.025,2820,48351,False,High,середній,велика
2014-05-02,7853361370,555000.0,4,2.50,3310,6500,2.0,0,0,3,...,0,98065,47.5150,-121.870,2380,5000,False,Mid-High,середній,велика
2014-05-02,5056500260,440000.0,4,2.25,2160,8119,1.0,0,0,3,...,0,98006,47.5443,-122.177,1850,9000,False,Mid-Low,середній,середня
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2015-05-14,1422700040,183000.0,3,1.00,1170,7320,1.0,0,0,3,...,0,98188,47.4685,-122.282,2040,7320,False,Low,середній,мала
2015-05-14,8730000270,359000.0,2,2.75,1370,1140,2.0,0,0,3,...,0,98133,47.7052,-122.343,1370,1090,False,Mid-Low,середній,мала
2015-05-15,7923600250,450000.0,5,2.00,1870,7344,1.5,0,0,3,...,0,98007,47.5951,-122.144,1870,7650,False,Mid-Low,середній,середня


In [104]:
df.area_category.value_counts()

,count
area_category,
середній,19336
малий,1494
великий,783


In [105]:
# Альтернатива: qcut() — автоматичне розбиття на квантилі
# Корисно, коли хочемо однакову кількість спостережень у групі, а не однакову ширину інтервалу.

df['area_auto_category'] = pd.qcut(
    df['sqft_living'],
    q=3,
    labels=['мала', 'середня', 'велика'],
)
df

,id,price,bedrooms,bathrooms,sqft_living,sqft_lot,floors,waterfront,view,condition,...,yr_renovated,zipcode,lat,long,sqft_living15,sqft_lot15,is_renovated,price_segment,area_category,area_auto_category
date,,,,,,,,,,,,,,,,,,,,,
2014-05-02,5561000190,437500.0,3,2.25,1970,35100,2.0,0,0,4,...,0,98027,47.4635,-121.991,2340,35100,False,Mid-Low,середній,середня
2014-05-02,472000620,790000.0,3,2.50,2600,4750,1.0,0,0,4,...,0,98117,47.6833,-122.400,2380,4750,False,High,середній,велика
2014-05-02,1024069009,675000.0,5,2.50,2820,67518,2.0,0,0,3,...,0,98029,47.5794,-122.025,2820,48351,False,High,середній,велика
2014-05-02,7853361370,555000.0,4,2.50,3310,6500,2.0,0,0,3,...,0,98065,47.5150,-121.870,2380,5000,False,Mid-High,середній,велика
2014-05-02,5056500260,440000.0,4,2.25,2160,8119,1.0,0,0,3,...,0,98006,47.5443,-122.177,1850,9000,False,Mid-Low,середній,середня
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2015-05-14,1422700040,183000.0,3,1.00,1170,7320,1.0,0,0,3,...,0,98188,47.4685,-122.282,2040,7320,False,Low,середній,мала
2015-05-14,8730000270,359000.0,2,2.75,1370,1140,2.0,0,0,3,...,0,98133,47.7052,-122.343,1370,1090,False,Mid-Low,середній,мала
2015-05-15,7923600250,450000.0,5,2.00,1870,7344,1.5,0,0,3,...,0,98007,47.5951,-122.144,1870,7650,False,Mid-Low,середній,середня


In [106]:
df.area_auto_category.value_counts()

,count
area_auto_category,
мала,7233
велика,7192
середня,7188


## Навіщо аналітику `cut` і `qcut`

| Задача | Рішення |
|---|---|
| Побудова сегментів | `cut` або `qcut` |
| Когортний аналіз (покоління, ціна) | `cut` по даті або ціні |
| Спрощення числових фіч | категоризація |
| Візуалізація (pie, bar, boxplot) | категорії зручніші |

## Типові помилки

| Проблема | Рішення |
|---|---|
| Значення на межі не включене | `include_lowest=True` |
| Некоректні категорії | задати `labels` вручну або перевірити |
| Багато унікальних значень у `cut` | замість `cut` — використовувати `qcut` |
| Нерівномірні категорії | вручну задати `bins`, якщо треба контроль |

---

# Підсумок: що коли використовувати

| Інструмент | Коли використовувати |
|---|---|
| `groupby()` + `.agg()` | Потрібні метрики в розрізі однієї або кількох категорій, результат — "довга" таблиця |
| `pivot_table()` | Та сама агрегація, але зручніше дивитись як матрицю "категорія А по рядках × категорія Б по стовпцях" |
| `crosstab()` | Цікавить саме **кількість спостережень** (чи частки) для комбінацій категорій, а не агрегація метрики |
| `resample()` | Групування за **часом** (дні/тижні/місяці), а не за категоріями |
| `cut()` / `qcut()` | Числову колонку треба спочатку перетворити на категорії, перш ніж групувати чи візуалізувати |

### Три головні звички, які варто винести з цієї лекції

1. **Якщо колонка `Categorical` і бере участь у `groupby()` — пишіть `observed=True` явно.** Інакше можна отримати рядки з `NaN` для комбінацій, яких ніколи не було в даних, і сплутати їх із реальними пропусками.
2. **Після `groupby()`/`agg()`/`pivot_table()` перевіряйте форму результату** (`MultiIndex` по колонках чи рядках) — `.reset_index()` і `.T` часто потрібні, щоб з результатом можна було зручно працювати далі чи будувати графік.
3. **У `resample()` використовуйте сучасні алиаси частот** (`'ME'`, `'QE'`, `'h'`, `'min'`, `'s'`) — старі однолітерні варіанти (`'M'`, `'Q'`, `'H'`, `'T'`) вже застарілі й з часом перестануть працювати.